In [55]:
import pandas as pd
import os
import numpy as np

In [56]:
input_filepath = r"C:\Users\marie\rep_codes\udder_project\udder_processing\features_dict\feature_table.csv"
output_path = r"C:\Users\marie\rep_codes\udder_project\udder_processing\features_dict"
mising_path  = r"C:\Users\marie\rep_codes\udder_project\delpro_vms\data\missing_teats.csv"

In [57]:
dtypes = {'cow':'str', 'frame':'Int64', 'filename':'str', 'udder_vol': np.float64, 'lf_vol':np.float64, 'rf_vol':np.float64, 'lb_vol':np.float64, \
 'rb_vol':np.float64, 'udder_sarea':np.float64, 'lf_sarea':np.float64, 'rf_sarea':np.float64, 'lb_sarea':np.float64, 'rb_sarea':np.float64, \
 'lf_angle':np.float64, 'rf_angle':np.float64, 'lb_angle':np.float64, 'rb_angle':np.float64, 'front_eu':np.float64, 'back_eu':np.float64, \
 'right_eu':np.float64, 'left_eu':np.float64, 'front_gd':np.float64, 'back_gd':np.float64, 'right_gd':np.float64, 'left_gd':np.float64,\
 'udder_peri':np.float64, 'udder_area':np.float64, 'udder_circ':np.float64, 'udder_exc':np.float64, 'lf_peri':np.float64, 'lf_area':np.float64, \
 'lf_circ':np.float64, 'lf_exc':np.float64, 'rf_peri':np.float64, 'rf_area':np.float64, 'rf_circ':np.float64, 'rf_exc':np.float64, 'lb_peri':np.float64, \
 'lb_area':np.float64, 'lb_circ':np.float64, 'lb_exc':np.float64, 'rb_peri':np.float64, 'rb_area':np.float64, 'rb_circ':np.float64, 'rb_exc':np.float64, \
 'lf_len':np.float64, 'rf_len':np.float64, 'lb_len':np.float64, 'rb_len':np.float64}

In [58]:
fdf = pd.read_csv(input_filepath)
fdf[fdf.columns[3:]] = fdf[fdf.columns[3:]].apply(pd.to_numeric, errors='coerce')

mt_df = pd.read_csv(mising_path)
cow_list = np.unique(mt_df.cow)
len(cow_list)

C:\Users\marie\AppData\Local\Temp\ipykernel_73012\3536355538.py:1: DtypeWarning: Columns (17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  fdf = pd.read_csv(input_filepath)


30

In [59]:
# cange values of missing teats to missing
for cow in cow_list:
    teats_missing = mt_df[mt_df.cow == cow]["teat"].values
    for teat in teats_missing:
        columns_with_teat = [col for col in fdf.columns if teat in col]
        fdf.loc[fdf.cow == cow, columns_with_teat] = np.nan
        temp = fdf[fdf.cow == cow][columns_with_teat]
        # print(temp)
    fdf.to_csv(os.path.join(feature_path, "mfeature_table.csv"))

In [44]:
fdfg = fdf.drop(["frame", "filename"], axis = 1).groupby(["cow"]).median()

fdfg_cols = fdfg.columns
for colname in fdfg_cols:
    # print(colname)
    Q1 = np.nanpercentile(fdfg[colname], 25, method='midpoint')
    Q3 = np.nanpercentile(fdfg[colname], 75, method='midpoint')
    med = np.nanpercentile(fdfg[colname], 50, method='midpoint')
    IQR = Q3 - Q1
    low_thres = med - 1.5*IQR
    up_thres = med + 1.5*IQR
    out = (fdfg[colname] < low_thres)| (fdfg[colname] > up_thres)
    fdfg.loc[out, colname] = np.nan

fdfg = fdfg.reset_index()
fdfg.to_csv(os.path.join(feature_path, "gmfeature_table.csv"))

In [62]:
fdf[fdf.columns[3:]] = fdf[fdf.columns[3:]].apply(pd.to_numeric, errors='coerce')
fdfg = fdf.drop(["frame", "filename"], axis = 1).groupby(["cow"]).median()
fdfg2 = fdfg.copy()
fdfg_cols = fdfg.columns

for colname in fdfg_cols:
    # print(colname)
    Q1 = np.nanpercentile(fdfg[colname], 25, method='midpoint')
    Q3 = np.nanpercentile(fdfg[colname], 75, method='midpoint')
    med = np.nanpercentile(fdfg[colname], 50, method='midpoint')
    IQR = Q3 - Q1
    low_thres = med - 1.5*IQR
    up_thres = med + 1.5*IQR
    out = (fdfg[colname] < low_thres)| (fdfg[colname] > up_thres)
    fdfg.loc[out, colname] = np.nan
    fdfg2.loc[out, colname] = np.nanmean(fdfg[colname])
fdfg = fdfg.reset_index()
fdfg.to_csv(os.path.join(feature_path, "gmfeature_table.csv"))


In [72]:
cow_list2 = list(set(cow_list).intersection(set(fdfg2.index)))
fdfg2["missing"] = 0
fdfg2.loc[cow_list2, "missing"] = 1

fdfg2.to_csv(os.path.join(feature_path, "gmfeature_table_cluster.csv"))